## UNSW-NB15 — Neural Network Multi-Task Classification

### Targets
- **`label`**: binary classification (0 = normal, 1 = attack)
- **`attack_cat`**: multi-class classification (type of attack)

### Architecture
- Deep feedforward neural network (MLP) with BatchNorm, Dropout, and L2 regularisation
- Reuses the same saved feature list and preprocessing assumptions from the RF pipeline
- Fits preprocessing on the training split only to avoid leakage
- Balances only the training fold so validation and test metrics stay honest
- Saves trained models compatible with the existing project structure


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight

from tensorflow import keras
from tensorflow.keras import callbacks, layers, regularizers

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# ── Paths ──────────────────────────────────────────────────────────────────
if Path('Dataset').exists():
    DATA_DIR = Path('Dataset')
    OUT_DIR  = Path('Processed_Dataset')
else:
    DATA_DIR = Path('training/Dataset')
    OUT_DIR  = Path('training/Processed_Dataset')

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR:', DATA_DIR.resolve())
print('OUT_DIR :', OUT_DIR.resolve())
print('TF version:', tf.__version__)

DATA_DIR: C:\Users\rhmar\GitHub\CTD-Project\training\Dataset
OUT_DIR : C:\Users\rhmar\GitHub\CTD-Project\training\Processed_Dataset
TF version: 2.19.0


### 1) Load pre-processed dataset
We reuse the `UNSW_NB15_processed.csv` produced by the RF notebook, and
the `selected_features.json` / `dropped_features.json` metadata.

In [2]:
PROCESSED_CSV = OUT_DIR / 'UNSW_NB15_processed.csv'
FEATURES_JSON = OUT_DIR / 'selected_features.json'
DROPPED_JSON  = OUT_DIR / 'dropped_features.json'

assert PROCESSED_CSV.exists(), (
    f'Run the RF notebook first to generate: {PROCESSED_CSV}'
)
assert FEATURES_JSON.exists(), (
    f'Missing saved feature metadata: {FEATURES_JSON}'
)

df = pd.read_csv(PROCESSED_CSV, low_memory=False)
feature_cols = json.loads(FEATURES_JSON.read_text())
dropped_feature_log = json.loads(DROPPED_JSON.read_text()) if DROPPED_JSON.exists() else []

# Remove helper column added in RF notebook if present
if 'is_dos' in df.columns:
    df = df.drop(columns=['is_dos'])
    feature_cols = [c for c in feature_cols if c != 'is_dos']

TARGET_LABEL  = 'label'
TARGET_ATTACK = 'attack_cat'

missing_features = sorted(set(feature_cols) - set(df.columns))
assert not missing_features, f'Missing expected features: {missing_features}'

MODEL_FEATURES = feature_cols.copy()

print('Dataset shape :', df.shape)
print('Features      :', len(MODEL_FEATURES))
print('Using the exact saved feature list from the RF pipeline.')
if dropped_feature_log:
    print('\nDropped during RF preprocessing:')
    for item in dropped_feature_log:
        print(f"- {item['columns']} -> {item['reason']}")

print('\nAttack distribution:')
print(df[TARGET_ATTACK].value_counts())

Dataset shape : (2059414, 40)
Features      : 38
Using the exact saved feature list from the RF pipeline.

Attack distribution:
attack_cat
Normal            1959771
Exploits            27599
Generic             25378
Fuzzers             21795
Reconnaissance      13357
DoS                  5665
Analysis             2184
Backdoor             1983
Shellcode            1511
Worms                 171
Name: count, dtype: int64


### 2) Validate attack-relevant feature coverage

We keep the RF notebook's selected feature list, then confirm it still contains the most useful network-security signals: ports, protocol/state, duration, packet-size statistics, and flow counters.

In [3]:
FEATURE_GROUPS = {
    'ports': ['sport', 'dsport'],
    'protocol': ['proto', 'state', 'service'],
    'duration': ['dur'],
    'packet_size': ['sbytes', 'dbytes', 'spkts', 'smeansz', 'dmeansz'],
    'flow_based': [
        'sload', 'dload', 'ct_state_ttl', 'ct_srv_src', 'ct_srv_dst',
        'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
        'ct_dst_src_ltm',
    ],
}

feature_audit = {
    group: [col for col in cols if col in MODEL_FEATURES]
    for group, cols in FEATURE_GROUPS.items()
}

print('Model feature count:', len(MODEL_FEATURES))
for group, cols in feature_audit.items():
    print(f'{group:12s}: {cols}')

missing_priority = sorted({
    col
    for cols in FEATURE_GROUPS.values()
    for col in cols
    if col not in MODEL_FEATURES
})

if missing_priority:
    print('\nPriority features not present in the saved feature list:')
    print(missing_priority)
else:
    print('\nAll priority attack-oriented feature groups are represented.')

Model feature count: 38
ports       : ['sport', 'dsport']
protocol    : ['proto', 'state', 'service']
duration    : ['dur']
packet_size : ['sbytes', 'dbytes', 'spkts', 'smeansz', 'dmeansz']
flow_based  : ['sload', 'dload', 'ct_state_ttl', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm']

All priority attack-oriented feature groups are represented.


### 3) Rebuild the preprocessing pipeline
We mirror the RF notebook's preprocessing contract: median imputation for numeric columns, most-frequent imputation plus ordinal encoding for categorical columns, then `StandardScaler` so the DNN receives normalized inputs.

The preprocessor is fit on the training split only.

In [4]:
# Detect column types from the saved feature list
KNOWN_CAT = {'proto', 'state', 'service'}
numeric_features     = [c for c in MODEL_FEATURES if c not in KNOWN_CAT]
categorical_features = [c for c in MODEL_FEATURES if c in KNOWN_CAT]

print('Numeric features    :', len(numeric_features))
print('Categorical features:', categorical_features)

encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1,
    encoded_missing_value=-1,
)

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', encoder),
    ('scaler', StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop',
)

Numeric features    : 35
Categorical features: ['proto', 'state', 'service']


### 4) Split first, then fit the preprocessor
This keeps validation and test data untouched by oversampling, imputation fitting, encoding fitting, and scaling statistics.

In [5]:
X_raw = df[MODEL_FEATURES].copy()
y_label = df[TARGET_LABEL].astype(int)
y_attack_name = df[TARGET_ATTACK].astype(str)

(
    X_train_raw, X_temp_raw,
    y_lbl_train, y_lbl_temp,
    y_atk_train_name, y_atk_temp_name,
) = train_test_split(
    X_raw,
    y_label,
    y_attack_name,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_attack_name,
)

(
    X_val_raw, X_test_raw,
    y_lbl_val, y_lbl_test,
    y_atk_val_name, y_atk_test_name,
) = train_test_split(
    X_temp_raw,
    y_lbl_temp,
    y_atk_temp_name,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_atk_temp_name,
)

le_attack = LabelEncoder()
y_atk_train = le_attack.fit_transform(y_atk_train_name)
y_atk_val = le_attack.transform(y_atk_val_name)
y_atk_test = le_attack.transform(y_atk_test_name)
NUM_CLASSES = len(le_attack.classes_)
print('Attack classes:', list(le_attack.classes_))

# Fit ONLY on the training fold to avoid leakage
X_train = preprocessor.fit_transform(X_train_raw).astype('float32')
X_val = preprocessor.transform(X_val_raw).astype('float32')
X_test = preprocessor.transform(X_test_raw).astype('float32')

split_summary = pd.DataFrame({
    'rows': [len(X_train_raw), len(X_val_raw), len(X_test_raw)],
    'attack_rate': [
        float(y_lbl_train.mean()),
        float(y_lbl_val.mean()),
        float(y_lbl_test.mean()),
    ],
}, index=['train', 'validation', 'test'])

display(split_summary)
print('X_train shape:', X_train.shape)
print('X_val   shape:', X_val.shape)
print('X_test  shape:', X_test.shape)

Attack classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']


,rows,attack_rate
train,1441589,0.048384
validation,308912,0.048383
test,308913,0.048386


X_train shape: (1441589, 38)
X_val   shape: (308912, 38)
X_test  shape: (308913, 38)


### 5) Handle imbalance on the training fold only

- Binary task: keep the original train distribution but apply class weights.
- Multi-class task: rebalance the training fold so each attack category gets equal representation, while validation and test sets stay untouched.

In [6]:
def rebalance_training_frame(X_frame: pd.DataFrame, y_values: pd.Series, target_size: int, target_name: str):
    work = X_frame.copy()
    work[target_name] = pd.Series(y_values, index=X_frame.index).values

    balanced_parts = []
    summary_rows = []
    for class_name, group in work.groupby(target_name):
        replace = len(group) < target_size
        sampled = resample(
            group,
            replace=replace,
            n_samples=target_size,
            random_state=RANDOM_STATE,
        )
        balanced_parts.append(sampled)
        summary_rows.append({
            target_name: class_name,
            'original_rows': len(group),
            'target_rows': target_size,
            'sampling': 'upsample' if len(group) < target_size else ('downsample' if len(group) > target_size else 'keep'),
        })

    balanced = (
        pd.concat(balanced_parts, axis=0)
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
    summary = pd.DataFrame(summary_rows).sort_values(target_name).reset_index(drop=True)
    return balanced, summary


lbl_classes = np.unique(y_lbl_train)
lbl_weights = compute_class_weight(class_weight='balanced', classes=lbl_classes, y=y_lbl_train)
label_class_weight = dict(zip(lbl_classes.tolist(), lbl_weights.tolist()))
print('Binary class weights:', label_class_weight)

attack_class_counts = y_atk_train_name.value_counts()
attack_target_size = int(
    attack_class_counts.drop(labels='Normal', errors='ignore').max()
    if len(attack_class_counts.drop(labels='Normal', errors='ignore'))
    else attack_class_counts.max()
)

attack_train_df, attack_balance_report = rebalance_training_frame(
    X_train_raw,
    y_atk_train_name,
    target_size=attack_target_size,
    target_name=TARGET_ATTACK,
)

X_train_attack = preprocessor.transform(attack_train_df[MODEL_FEATURES]).astype('float32')
y_train_attack = le_attack.transform(attack_train_df[TARGET_ATTACK])

print('\nBalanced attack-category training distribution:')
display(attack_balance_report)
print('Balanced multi-class train shape:', X_train_attack.shape)

Binary class weights: {0: 0.5254220794131089, 1: 10.333971326164875}

Balanced attack-category training distribution:


,attack_cat,original_rows,target_rows,sampling
0,Analysis,1529,19319,upsample
1,Backdoor,1388,19319,upsample
2,DoS,3965,19319,upsample
3,Exploits,19319,19319,keep
4,Fuzzers,15256,19319,upsample
5,Generic,17765,19319,upsample
6,Normal,1371839,19319,downsample
7,Reconnaissance,9350,19319,upsample
8,Shellcode,1058,19319,upsample
9,Worms,120,19319,upsample


Balanced multi-class train shape: (193190, 38)


### 6) Neural Network architecture

```
Input → Dense(256) → BN → ReLU → Dropout(0.35)
      → Dense(128) → BN → ReLU → Dropout(0.25)
      → Dense(64)  → BN → ReLU → Dropout(0.20)
      → Dense(32)  → BN → ReLU → Dropout(0.10)
      → Output
```

This version adds a fourth hidden layer, L2 regularisation, explicit validation callbacks, and a smaller learning rate for more stable convergence.

In [7]:
TRAINING_CONFIG = {
    'epochs': 80,
    'batch_size': 256,
    'learning_rate': 3e-4,
    'l2_strength': 1e-4,
}


def build_model(input_dim: int, output_dim: int, task: str, learning_rate: float = TRAINING_CONFIG['learning_rate']) -> keras.Model:
    """Build a regularized deep MLP for binary or multi-class classification."""
    inp = keras.Input(shape=(input_dim,), name='features')
    x = inp

    for units, dropout_rate in [(256, 0.35), (128, 0.25), (64, 0.20), (32, 0.10)]:
        x = layers.Dense(
            units,
            kernel_initializer='he_normal',
            kernel_regularizer=regularizers.l2(TRAINING_CONFIG['l2_strength']),
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout_rate)(x)

    if task == 'binary':
        out = layers.Dense(1, activation='sigmoid', name='output')(x)
        loss = 'binary_crossentropy'
        metrics = [
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='auc'),
        ]
    else:
        out = layers.Dense(output_dim, activation='softmax', name='output')(x)
        loss = 'sparse_categorical_crossentropy'
        metrics = [keras.metrics.SparseCategoricalAccuracy(name='accuracy')]

    model = keras.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=loss,
        metrics=metrics,
    )
    return model


def make_callbacks() -> list[callbacks.Callback]:
    return [
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


def plot_training_curves(history: keras.callbacks.History, metric_name: str, title_prefix: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['loss'], label='train')
    axes[0].plot(history.history['val_loss'], label='val')
    axes[0].set_title(f'{title_prefix} loss')
    axes[0].legend()

    axes[1].plot(history.history[metric_name], label='train')
    axes[1].plot(history.history[f'val_{metric_name}'], label='val')
    axes[1].set_title(f'{title_prefix} {metric_name}')
    axes[1].legend()
    plt.tight_layout()
    plt.show()


INPUT_DIM = X_train.shape[1]
print('Input dimension:', INPUT_DIM)
print('Training config:', TRAINING_CONFIG)

Input dimension: 38
Training config: {'epochs': 80, 'batch_size': 256, 'learning_rate': 0.0003, 'l2_strength': 0.0001}


### 7) Task A — Binary classification (`label`)

In [ ]:
print('=' * 60)
print('Task A  —  Binary label classification')
print('=' * 60)

model_label = build_model(INPUT_DIM, output_dim=1, task='binary')
model_label.summary()

history_label = model_label.fit(
    X_train,
    y_lbl_train,
    validation_data=(X_val, y_lbl_val),
    epochs=TRAINING_CONFIG['epochs'],
    batch_size=TRAINING_CONFIG['batch_size'],
    class_weight=label_class_weight,
    callbacks=make_callbacks(),
    verbose=1,
)

Task A  —  Binary label classification


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 38)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │         9,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 55,169 (215.50 KB)

 Trainable params: 54,209 (211.75 KB)

 Non-trainable params: 960 (3.75 KB)

Epoch 1/80
5632/5632 ━━━━━━━━━━━━━━━━━━━━ 34s 5ms/step - accuracy: 0.9451 - auc: 0.9875 - loss: 0.2187 - precision: 0.5372 - recall: 0.9907 - val_accuracy: 0.9855 - val_auc: 0.9977 - val_loss: 0.0950 - val_precision: 0.7691 - val_recall: 1.0000 - learning_rate: 3.0000e-04
Epoch 2/80
5632/5632 ━━━━━━━━━━━━━━━━━━━━ 32s 6ms/step - accuracy: 0.9849 - auc: 0.9968 - loss: 0.0764 - precision: 0.7624 - recall: 0.9992 - val_accuracy: 0.9856 - val_auc: 0.9980 - val_loss: 0.0689 - val_precision: 0.7704 - val_recall: 0.9999 - learning_rate: 3.0000e-04
Epoch 3/80
5632/5632 ━━━━━━━━━━━━━━━━━━━━ 29s 5ms/step - accuracy: 0.9853 - auc: 0.9974 - loss: 0.0539 - precision: 0.7671 - recall: 0.9995 - val_accuracy: 0.9856 - val_auc: 0.9982 - val_loss: 0.0592 - val_precision: 0.7710 - val_recall: 1.0000 - learning_rate: 3.0000e-04
Epoch 4/80
5632/5632 ━━━━━━━━━━━━━━━━━━━━ 30s 5ms/step - accuracy: 0.9854 - auc: 0.9977 - loss: 0.0455 - precision: 0.7677 - recall: 0.9994 - val_accuracy: 0.9856 - val_auc: 0.9982 

In [ ]:
# ── Evaluate Task A ─────────────────────────────────────────────────────────
y_lbl_val_prob = model_label.predict(X_val, batch_size=TRAINING_CONFIG['batch_size']).ravel()
threshold_grid = np.arange(0.20, 0.81, 0.05)
threshold_rows = []

for threshold in threshold_grid:
    y_val_pred = (y_lbl_val_prob >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'precision_attack': precision_score(y_lbl_val, y_val_pred, zero_division=0),
        'recall_attack': recall_score(y_lbl_val, y_val_pred, zero_division=0),
        'f1_attack': f1_score(y_lbl_val, y_val_pred, zero_division=0),
        'f2_attack': fbeta_score(y_lbl_val, y_val_pred, beta=2, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows).sort_values(
    ['f2_attack', 'recall_attack', 'precision_attack'],
    ascending=False,
).reset_index(drop=True)
best_threshold = float(threshold_df.loc[0, 'threshold'])

print('\nValidation threshold search (top 10):')
display(threshold_df.head(10))
print(f'Selected threshold for attack recall/F2 balance: {best_threshold:.2f}')

y_lbl_prob = model_label.predict(X_test, batch_size=TRAINING_CONFIG['batch_size']).ravel()
y_lbl_pred = (y_lbl_prob >= best_threshold).astype(int)

binary_metrics = {
    'accuracy': accuracy_score(y_lbl_test, y_lbl_pred),
    'precision_attack': precision_score(y_lbl_test, y_lbl_pred, zero_division=0),
    'recall_attack': recall_score(y_lbl_test, y_lbl_pred, zero_division=0),
    'f1_attack': f1_score(y_lbl_test, y_lbl_pred, zero_division=0),
}

print('\n--- Task A | Neural Network | Test metrics ---')
for metric_name, metric_value in binary_metrics.items():
    print(f'{metric_name}: {metric_value:.6f}')
print(classification_report(
    y_lbl_test,
    y_lbl_pred,
    target_names=['Normal', 'Attack'],
    digits=4,
    zero_division=0,
))

cm_a = confusion_matrix(y_lbl_test, y_lbl_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_a,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Normal', 'Attack'],
    yticklabels=['Normal', 'Attack'],
)
plt.title('Task A — Confusion matrix (Neural Network)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

plot_training_curves(history_label, metric_name='accuracy', title_prefix='Binary task')

### 8) Task B — Multi-class classification (`attack_cat`)

In [ ]:
print('=' * 60)
print('Task B  —  Attack category classification')
print('=' * 60)

model_attack = build_model(INPUT_DIM, output_dim=NUM_CLASSES, task='multiclass')
model_attack.summary()

history_attack = model_attack.fit(
    X_train_attack,
    y_train_attack,
    validation_data=(X_val, y_atk_val),
    epochs=TRAINING_CONFIG['epochs'],
    batch_size=TRAINING_CONFIG['batch_size'],
    callbacks=make_callbacks(),
    verbose=1,
)

In [ ]:
# ── Evaluate Task B ─────────────────────────────────────────────────────────
y_atk_prob = model_attack.predict(X_test, batch_size=TRAINING_CONFIG['batch_size'])
y_atk_pred = np.argmax(y_atk_prob, axis=1)

y_atk_test_names = le_attack.inverse_transform(y_atk_test)
y_atk_pred_names = le_attack.inverse_transform(y_atk_pred)
class_labels = sorted(le_attack.classes_)

attack_test_accuracy = accuracy_score(y_atk_test, y_atk_pred)
attack_macro_recall = recall_score(y_atk_test, y_atk_pred, average='macro', zero_division=0)
attack_macro_f1 = f1_score(y_atk_test, y_atk_pred, average='macro', zero_division=0)

print('\n--- Task B | Neural Network | Test metrics ---')
print(f'accuracy:     {attack_test_accuracy:.6f}')
print(f'macro recall: {attack_macro_recall:.6f}')
print(f'macro f1:     {attack_macro_f1:.6f}')
print(classification_report(
    y_atk_test_names,
    y_atk_pred_names,
    labels=class_labels,
    digits=4,
    zero_division=0,
))

attack_report = pd.DataFrame(
    classification_report(
        y_atk_test_names,
        y_atk_pred_names,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
).T

print('Per-class precision/recall/F1 (sorted by recall):')
display(attack_report.loc[class_labels, ['precision', 'recall', 'f1-score', 'support']].sort_values('recall'))

cm_b = confusion_matrix(y_atk_test_names, y_atk_pred_names, labels=class_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_b,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels,
)
plt.title('Task B — Confusion matrix (Neural Network)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plot_training_curves(history_attack, metric_name='accuracy', title_prefix='Attack-category task')

### 9) Save models & summaries

In [ ]:
# ── Save Keras models (native format) ───────────────────────────────────────
label_keras_path  = OUT_DIR / 'model_label_nn.keras'
attack_keras_path = OUT_DIR / 'model_attack_cat_nn.keras'

model_label.save(str(label_keras_path))
model_attack.save(str(attack_keras_path))
print('Saved:', label_keras_path)
print('Saved:', attack_keras_path)

# ── Save the sklearn preprocessor so we can reproduce transforms ─────────────
preproc_path = OUT_DIR / 'nn_preprocessor.joblib'
joblib.dump(preprocessor, preproc_path)
print('Saved:', preproc_path)

# ── Save label encoder for attack_cat ───────────────────────────────────────
le_path = OUT_DIR / 'nn_label_encoder_attack.joblib'
joblib.dump(le_attack, le_path)
print('Saved:', le_path)

# ── Save summary JSON (mirrors RF notebook format) ───────────────────────────
nn_label_cv = float(max(history_label.history['val_accuracy']))
nn_attack_cv = float(max(history_attack.history['val_accuracy']))

label_summary = {
    'task': 'label',
    'selected_model': 'NeuralNetwork (MLP)',
    'best_val_accuracy': round(nn_label_cv, 6),
    'best_threshold': round(best_threshold, 4),
    'test_accuracy': round(float(binary_metrics['accuracy']), 6),
    'test_precision_attack': round(float(binary_metrics['precision_attack']), 6),
    'test_recall_attack': round(float(binary_metrics['recall_attack']), 6),
    'test_f1_attack': round(float(binary_metrics['f1_attack']), 6),
    'training_config': TRAINING_CONFIG,
    'features': MODEL_FEATURES,
}
attack_summary = {
    'task': 'attack_cat',
    'selected_model': 'NeuralNetwork (MLP)',
    'best_val_accuracy': round(nn_attack_cv, 6),
    'test_accuracy': round(float(attack_test_accuracy), 6),
    'test_macro_recall': round(float(attack_macro_recall), 6),
    'test_macro_f1': round(float(attack_macro_f1), 6),
    'balanced_train_target_per_class': attack_target_size,
    'training_config': TRAINING_CONFIG,
    'classes': list(le_attack.classes_),
    'features': MODEL_FEATURES,
}

(OUT_DIR / 'model_label_nn_summary.json').write_text(
    json.dumps(label_summary, indent=2)
)
(OUT_DIR / 'model_attack_cat_nn_summary.json').write_text(
    json.dumps(attack_summary, indent=2)
)
print('Summaries saved.')

### 10) Inference helper — how to load & use the NN

```python
import joblib, json
import numpy as np
import tensorflow as tf
from pathlib import Path

OUT_DIR = Path('training/Processed_Dataset')

preproc  = joblib.load(OUT_DIR / 'nn_preprocessor.joblib')
le_atk   = joblib.load(OUT_DIR / 'nn_label_encoder_attack.joblib')
nn_label = tf.keras.models.load_model(OUT_DIR / 'model_label_nn.keras')
nn_atk   = tf.keras.models.load_model(OUT_DIR / 'model_attack_cat_nn.keras')
label_meta = json.loads((OUT_DIR / 'model_label_nn_summary.json').read_text())
feature_cols = label_meta['features']
threshold = label_meta['best_threshold']

# Given a raw DataFrame `new_df` with the saved selected feature columns:
X_new = preproc.transform(new_df[feature_cols]).astype('float32')

label_prob = nn_label.predict(X_new).ravel()            # P(attack)
label_pred = (label_prob >= threshold).astype(int)      # 0=Normal / 1=Attack

attack_prob = nn_atk.predict(X_new)
attack_pred = le_atk.inverse_transform(np.argmax(attack_prob, axis=1))
```

### 11) Further improvement ideas

- Compare simple resampling against `SMOTE` or `ADASYN` after preprocessing to see whether synthetic minority examples improve tail-class recall.
- Tune the binary decision threshold based on your operational goal: higher attack recall for detection, or higher precision to reduce false alarms.
- Try a shared multi-task network with two output heads so `label` and `attack_cat` learn from the same traffic representation.
- Add PR-AUC, per-class recall tracking, and a time-aware split if your traffic captures are chronological rather than IID.
- If the model still overfits, reduce layer width or increase dropout before adding more depth.